### **TIDIGITS DTW**

In [1]:
import numpy as np
from src.dtw import DTW
from src.pca import PCA


### **Load Data**

In [2]:
sequences = []
labels = []
for digit in [1, 5, 6]:
    with np.load(f"./data/tidigits/{digit}.npz") as data:
        for key in data.files:
            sequences.append(data[key])
            labels.append(digit)

y = np.array(labels)

print("Recordings:", len(sequences))
print("Labels:", y.shape)


Recordings: 1350
Labels: (1350,)


### **Shuffle and Split**

In [3]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(y))
train_end = int(0.6 * len(y))
val_end = int(0.8 * len(y))

train_indices = indices[:train_end]
val_indices = indices[train_end:val_end]
test_indices = indices[val_end:]

y_train = y[train_indices]
y_val = y[val_indices]
y_test = y[test_indices]


### **Padding**

In [4]:
raw_train_sequences = [sequences[i] for i in train_indices]
raw_val_sequences = [sequences[i] for i in val_indices]
raw_test_sequences = [sequences[i] for i in test_indices]
# frame_length = int(np.median([len(s) for s in raw_train_sequences]))
frame_length = int(np.quantile([len(s) for s in raw_train_sequences], q=0.95))



def pad_sequences(recordings):
    padded = []
    for recording in recordings:
        values = np.zeros((frame_length, recording.shape[1]))
        count = min(len(recording), frame_length)
        values[:count] = recording[:count]
        padded.append(values)
    return padded


train_sequences = pad_sequences(raw_train_sequences)
val_sequences = pad_sequences(raw_val_sequences)
test_sequences = pad_sequences(raw_test_sequences)
print("Target frames:", frame_length)


Target frames: 135


In [5]:
np.array(train_sequences).shape

In [6]:
classes = np.unique(y_train)
print("Class order:", classes)
print("Train recordings:", len(train_sequences))
print("Validation recordings:", len(val_sequences))
print("Test recordings:", len(test_sequences))
print("Recording shape:", train_sequences[0].shape)


Class order: [1 5 6]
Train recordings: 810
Validation recordings: 270
Test recordings: 270
Recording shape: (135, 39)


### **DTW**

In [7]:
dtw_model = DTW(references_per_class=10, window=25)
dtw_model.fit(train_sequences, y_train)
print("Reference recordings:", len(dtw_model.indices))
dtw_val_predictions = dtw_model.predict(val_sequences)
dtw_test_predictions = dtw_model.predict(test_sequences)


Reference recordings: 30


In [8]:
# Confusion matrix
dtw_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        dtw_confusion[i, j] = np.sum(
            (y_test == actual) & (dtw_test_predictions == predicted))

print("Validation accuracy:", np.mean(dtw_val_predictions == y_val))
print("Test accuracy:", np.mean(dtw_test_predictions == y_test))
print("Test confusion matrix:\n", dtw_confusion)


Validation accuracy: 0.8814814814814815
Test accuracy: 0.8703703703703703
Test confusion matrix:
 [[88  0  0]
 [19 75  0]
 [16  0 72]]


### **DTW with PCA**

In [9]:
K_dtw = 13
frame_pca = PCA(k=K_dtw)
frame_pca.fit(np.concatenate(raw_train_sequences).astype(float))
train_sequences_pca = [frame_pca.transform(s) for s in train_sequences]
val_sequences_pca = [frame_pca.transform(s) for s in val_sequences]
test_sequences_pca = [frame_pca.transform(s) for s in test_sequences]

print("First recording:", train_sequences[0].shape, "->", train_sequences_pca[0].shape)
dtw_pca_model = DTW(references_per_class=10, window=25)
dtw_pca_model.fit(train_sequences_pca, y_train)
print("Reference recordings:", len(dtw_pca_model.indices))
dtw_pca_val_predictions = dtw_pca_model.predict(val_sequences_pca)
dtw_pca_test_predictions = dtw_pca_model.predict(test_sequences_pca)


First recording: (135, 39) -> (135, 13)
Reference recordings: 30


In [10]:
# Confusion matrix
dtw_pca_confusion = np.zeros((len(classes), len(classes)), dtype=int)
for i, actual in enumerate(classes):
    for j, predicted in enumerate(classes):
        dtw_pca_confusion[i, j] = np.sum(
            (y_test == actual) & (dtw_pca_test_predictions == predicted))

print("Validation accuracy:", np.mean(dtw_pca_val_predictions == y_val))
print("Test accuracy:", np.mean(dtw_pca_test_predictions == y_test))
print("Test confusion matrix:\n", dtw_pca_confusion)


Validation accuracy: 0.9629629629629629
Test accuracy: 0.9629629629629629
Test confusion matrix:
 [[88  0  0]
 [ 8 86  0]
 [ 2  0 86]]
